In [ ]:
import boa
import os

pools = {
    "yb_cbBTC": "0x83f24023d15d835a213df24fd309c47dAb5BEb32",
    "yb_wBTC": "0xD9FF8396554A0d18B2CFbeC53e1979b7ecCe8373",
    "yb_tBTC": "0xf1F435B05D255a5dBdE37333C0f61DA6F69c6127",
}

lt_pools = {
    "yb_cbBTC": "0xD6a1147666f6E4d7161caf436d9923D44d901112",
    "yb_wBTC": "0x6095a220C5567360d459462A25b1AD5aEAD45204",
    "yb_tBTC": "0x2B513eBe7070Cff91cf699a0BFe5075020C732FF",
}
decimals = {
    "yb_cbBTC": 8,
    "yb_wBTC": 8,
    "yb_tBTC": 18,
}

In [ ]:
from abis import lt_abi
import json
from web3 import Web3

RPC_URL = os.environ.get("WEB3_PROVIDER_URL")
if not RPC_URL:
    raise RuntimeError("Set WEB3_PROVIDER_URL before running this script")
WEB3 = Web3(Web3.HTTPProvider(RPC_URL))

ABI = json.loads(lt_abi)

EVENT_NAMES = [entry.get("name", "") for entry in ABI if entry.get("type") == "event"]
print(EVENT_NAMES)

In [ ]:
# Pick pool and user to search for
pool = "yb_cbBTC"
user = "0xDF3E15D2335Be8190bA84E25568d6341af2bC0Be"
FROM_BLOCK = 23_434_000
TO_BLOCK = "latest"

# Build contract from LT (vault) ABI
lt_address = Web3.to_checksum_address(lt_pools[pool])
user_cs = Web3.to_checksum_address(user)
lt = WEB3.eth.contract(address=lt_address, abi=ABI)
logs_deposit = lt.events.Deposit().get_logs(
    from_block=FROM_BLOCK, to_block=TO_BLOCK, argument_filters={"sender": user_cs}
)
logs_withdraw = lt.events.Withdraw().get_logs(
    from_block=FROM_BLOCK, to_block=TO_BLOCK, argument_filters={"sender": user_cs}
)

In [ ]:
total_deposited = 0
deposit_ts = 0
for log in logs_deposit:
    total_deposited += log["args"]["assets"]
    if deposit_ts == 0:
        block_id = log["blockNumber"]
        deposit_ts = WEB3.eth.get_block(block_id)["timestamp"]
total_withdrawn = 0
for log in logs_withdraw:
    total_withdrawn += log["args"]["assets"]

total_outstanding = total_deposited - total_withdrawn
print(f"Total deposited: {total_deposited / 10**decimals[pool]} (at {deposit_ts})")
print(f"Total withdrawn: {total_withdrawn / 10**decimals[pool]}")
print(f"Total outstanding: {total_outstanding / 10**decimals[pool]}")

In [ ]:
# Setup boa
etherscan_api_key = os.environ.get("ETHERSCAN_API_KEY")
# boa.set_network_env(RPC_URL)
try:
    boa.fork(RPC_URL)
except Exception as e:
    print(f"Error forking: {e}")

acc = boa.env.generate_address()
boa.env.eoa = user
lt_boa = boa.load_partial("lt.vy").at(lt_address)
asset = lt_boa.ASSET_TOKEN()
asset_contract = boa.from_etherscan(asset, api_key=etherscan_api_key)
print(f"Underlying asset: {asset}")

In [ ]:
pre_asset_balance = asset_contract.balanceOf(user)
print(f"Current balance of asset: {pre_asset_balance}")
pre_lt_balance = lt_boa.balanceOf(user)
print(f"Current balance of LT: {pre_lt_balance}")

In [ ]:
with boa.env.prank(user):
    lt_boa.withdraw(pre_lt_balance, 0)
post_asset_balance = asset_contract.balanceOf(user)
print(f"Post-withdraw balance of asset: {post_asset_balance}")
post_lt_balance = lt_boa.balanceOf(user)
print(f"Post-withdraw balance of LT: {post_lt_balance}")

In [ ]:
total_withdrawn_full = total_withdrawn + post_asset_balance - pre_asset_balance
loss = (total_deposited - total_withdrawn_full) / total_deposited
print(f"Total deposit BTC: {total_deposited/10**decimals[pool]}")
print(f"Total withdraw BTC: {total_withdrawn_full/10**decimals[pool]}")
lp_duration = WEB3.eth.get_block("latest")["timestamp"] - deposit_ts
print(f"Loss: {100*loss:4.2f}%")